### Ejemplo 2 
Utilizando el dataset de "Used Cars" para construir, entrenar y evaluar una red neuronal utilizando Keras

In [1]:
#Primero, importamos las bibliotecas necesarias:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

#keras needs the four following aliases to be done manually or use python 3.9 or before
import collections.abc
collections.Iterable = collections.abc.Iterable
collections.Mapping = collections.abc.Mapping
collections.MutableSet = collections.abc.MutableSet
collections.MutableMapping = collections.abc.MutableMapping
#Now import keras

from keras.models import Sequential
from keras.layers import Dense

2023-03-17 16:49:46.147294: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


Luego, cargamos el conjunto de datos y realizamos algunas operaciones de limpieza de datos. Descargamos el archivo CSV de Kaggle y lo cargamos en un DataFrame de Pandas.

In [3]:
df = pd.read_csv('vehicles.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'vehicles.csv'

In [ ]:
df.head()

,id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,...,size,type,paint_color,image_url,description,county,state,lat,long,posting_date
0,7222695916,https://prescott.craigslist.org/cto/d/prescott...,prescott,https://prescott.craigslist.org,6000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,az,NaN,NaN,NaN
1,7218891961,https://fayar.craigslist.org/ctd/d/bentonville...,fayetteville,https://fayar.craigslist.org,11900,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ar,NaN,NaN,NaN
2,7221797935,https://keys.craigslist.org/cto/d/summerland-k...,florida keys,https://keys.craigslist.org,21000,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,fl,NaN,NaN,NaN
3,7222270760,https://worcester.craigslist.org/cto/d/west-br...,worcester / central MA,https://worcester.craigslist.org,1500,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,ma,NaN,NaN,NaN
4,7210384030,https://greensboro.craigslist.org/cto/d/trinit...,greensboro,https://greensboro.craigslist.org,4900,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,nc,NaN,NaN,NaN


Eliminamos las columnas que no se utilizarán en el análisis:

In [ ]:
df = df.drop(['id', 'url', 'region', 'region_url', 'image_url', 'description'], axis=1)

Convertimos la columna cylinders a tipo float y manejamos los valores nulos:

In [ ]:
df['cylinders'] = pd.to_numeric(df['cylinders'], errors='coerce')
df = df.drop(['cylinders','county'],axis=1)
df.head()
null_counts = df.isnull().sum()
#print(null_counts)

non_null_counts = df.count()
print(non_null_counts)


KeyError: 'cylinders'

#### Convertimos la columna fuel en una variable ficticia.

La función pd.get_dummies de la biblioteca Pandas se utiliza para convertir una variable categórica en varias variables numéricas binarias (0 o 1) que indican la presencia o ausencia de cada posible valor de la variable categórica. En este caso, la función se utiliza para convertir la columna 'fuel' en varias columnas numéricas binarias, una para cada valor único de 'fuel', lo que permitirá que estos valores sean utilizados como entradas en el modelo de red neuronal.

El parámetro columns se utiliza para especificar la(s) columna(s) que se deben convertir en variables dummy. En este caso, se está convirtiendo la columna 'fuel'.

In [ ]:
df = pd.get_dummies(df, columns=['fuel'])

Eliminamos los valores extremos en la columna price:

In [ ]:
df = df[df['price'].between(df['price'].quantile(.01), df['price'].quantile(.99))]
df.head()

,price,year,manufacturer,model,condition,odometer,title_status,transmission,VIN,drive,...,paint_color,state,lat,long,posting_date,fuel_diesel,fuel_electric,fuel_gas,fuel_hybrid,fuel_other
0,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,az,NaN,NaN,NaN,0,0,0,0,0
1,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,ar,NaN,NaN,NaN,0,0,0,0,0
2,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,fl,NaN,NaN,NaN,0,0,0,0,0
3,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,ma,NaN,NaN,NaN,0,0,0,0,0
4,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,nc,NaN,NaN,NaN,0,0,0,0,0


A continuación, separamos el conjunto de datos en conjuntos de entrenamiento y prueba utilizando train_test_split de Scikit-learn:

In [ ]:
price = df['price']
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(['price'], axis=1), price, train_size=0.8, test_size=0.2, random_state=0)

Normalizamos los datos de entrenamiento y prueba utilizando StandardScaler de Scikit-learn:

In [ ]:
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

ValueError: could not convert string to float: 'infiniti'

Luego, construimos el modelo de la red neuronal. En este ejemplo, utilizamos una red neuronal de 3 capas: una capa de entrada con 10 neuronas, una capa oculta con 6 neuronas y una capa de salida con 1 neurona. También utilizamos la función de activación relu para las dos primeras capas y la función de activación linear para la capa de salida.

In [ ]:
model = Sequential()
model.add(Dense(10, input_dim=X_train.shape[1], activation='relu'))
model.add(Dense(6, activation='relu'))
model.add(Dense(1, activation='linear'))

Compilamos el modelo utilizando el optimizador adam y la función de pérdida mean_squared_error.

In [ ]:
model.compile(loss='mean_squared_error', optimizer='adam')

Entrenamos el modelo utilizando los datos de entrenamiento:

In [ ]:
history = model.fit(X_train, y_train, validation_split=0.2,
                    epochs=50, batch_size=32, verbose=0)

Podemos visualizar el proceso de entrenamiento utilizando los datos almacenados en el objeto history. Este objeto contiene los valores de pérdida y precisión (exactitud) en cada época durante el entrenamiento.

Por ejemplo, si queremos visualizar la pérdida y precisión durante el entrenamiento de nuestra red neuronal, podemos usar el siguiente código:

In [ ]:
import matplotlib.pyplot as plt

# Obtener los valores de pérdida y precisión del objeto history
train_loss = history.history['loss']
val_loss = history.history['val_loss']
train_acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

# Graficar la pérdida
plt.plot(train_loss, label='train_loss')
plt.plot(val_loss, label='val_loss')
plt.title('Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Graficar la precisión
plt.plot(train_acc, label='train_acc')
plt.plot(val_acc, label='val_acc')
plt.title('Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

Después de haber entrenado nuestro modelo podemos hacer predicciones utilizando el método predict() del objeto modelo.

Por ejemplo, si queremos hacer predicciones para los datos de prueba, podemos utilizar el siguiente código:

In [ ]:
# Hacer predicciones para los datos de prueba
y_pred = model.predict(X_test)

# Imprimir las primeras 10 predicciones
print(y_pred[:10])